# Dynamic Quantization — GRU-LSTM Hybrid (C-MAPSS)
No `argparse` — runs cleanly inside Jupyter.

In [1]:
# ── ✏️  EDIT THESE TWO LINES ONLY ────────────────────────────────────────────
DATASET    = 'FD003'          # FD001 | FD002 | FD003 | FD004
DATA_DIR   = 'C:\\IIoT-Predictive-Maintenance\\processed'  # same as your notebook
# ─────────────────────────────────────────────────────────────────────────────

MODEL_PATH = f'{DATA_DIR}\\gru_lstm_{DATASET}.pt'
NPZ_PATH   = f'{DATA_DIR}\\{DATASET}_tensors.npz'

In [2]:
import os, copy, time
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [3]:
# ── Exact architecture from GRU_LSTM.ipynb ────────────────────────────────────

class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):               # x: (B, T, H)
        scores  = self.attn(x)          # (B, T, 1)
        weights = torch.softmax(scores, dim=1)
        context = (weights * x).sum(dim=1)
        return context, weights.squeeze(-1)


class GRULSTM(nn.Module):
    def __init__(self, input_size=18, gru_hidden=64, gru_layers=2,
                 lstm_hidden=64, lstm_layers=2, fc_hidden=32,
                 dropout=0.3, **kwargs):
        super().__init__()
        self.gru = nn.GRU(
            input_size, gru_hidden, gru_layers,
            batch_first=True,
            dropout=dropout if gru_layers > 1 else 0.0
        )
        self.gru_drop  = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            gru_hidden, lstm_hidden, lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0
        )
        self.lstm_drop = nn.Dropout(dropout)
        self.attention = TemporalAttention(lstm_hidden)
        self.fc = nn.Sequential(
            nn.Linear(lstm_hidden, fc_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fc_hidden, 1),
            nn.Sigmoid()
        )

    def forward(self, x, return_attn=False):
        gru_out,  _     = self.gru(x)
        gru_out         = self.gru_drop(gru_out)
        lstm_out, _     = self.lstm(gru_out)
        lstm_out        = self.lstm_drop(lstm_out)
        context, attn_w = self.attention(lstm_out)
        out             = self.fc(context)
        return (out, attn_w) if return_attn else out

print('✅ Model classes defined')

✅ Model classes defined


In [4]:
# ── Helper functions ──────────────────────────────────────────────────────────

def get_size_mb(model, tmp='_tmp.pt'):
    torch.save(model.state_dict(), tmp)
    mb = os.path.getsize(tmp) / (1024 ** 2)
    os.remove(tmp)
    return mb

def benchmark_ms(model, X, batch_size=64, runs=5):
    model.eval()
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(X)),
        batch_size=batch_size
    )
    times = []
    with torch.no_grad():
        for _ in range(runs):
            t0 = time.perf_counter()
            for (b,) in loader:
                model(b)
            times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times))

def predict(model, X, batch_size=64):
    model.eval()
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(X)),
        batch_size=batch_size
    )
    preds = []
    with torch.no_grad():
        for (b,) in loader:
            preds.append(model(b).numpy())
    return np.concatenate(preds).flatten()

def nasa_score(y_true, y_pred):
    d = y_pred - y_true
    return float(np.sum(np.where(d < 0, np.exp(-d/13)-1, np.exp(d/10)-1)))

print('✅ Helper functions defined')

✅ Helper functions defined


In [5]:
# ── Load checkpoint & rebuild model ──────────────────────────────────────────
print(f'Loading: {MODEL_PATH}')
ck      = torch.load(MODEL_PATH, map_location='cpu')
config  = ck['config']
max_rul = config['max_rul']
print(f'Config : {config}')

fp32_model = GRULSTM(**config).cpu().eval()
fp32_model.load_state_dict(ck['model_state'])
print(f'Params : {sum(p.numel() for p in fp32_model.parameters()):,}')
print('✅ FP32 model loaded')

Loading: C:\IIoT-Predictive-Maintenance\processed\gru_lstm_FD003.pt
Config : {'input_size': 18, 'gru_hidden': 64, 'gru_layers': 2, 'lstm_hidden': 64, 'lstm_layers': 2, 'fc_hidden': 32, 'dropout': 0.3, 'window_size': 30, 'max_rul': 125.0, 'dataset': 'FD003'}
Params : 111,874
✅ FP32 model loaded


In [6]:
# ── Load test tensors ─────────────────────────────────────────────────────────
print(f'Loading: {NPZ_PATH}')
npz    = np.load(NPZ_PATH)
X_test = npz['X_test'].astype(np.float32)   # (E, 30, 18)
y_test = npz['y_test'].astype(np.float32)   # (E,)

# Scale back to cycles if labels were saved normalised [0, 1]
if y_test.max() <= 1.0:
    y_test = y_test * max_rul

print(f'X_test : {X_test.shape}')
print(f'y_test : min={y_test.min():.0f}  max={y_test.max():.0f}  mean={y_test.mean():.1f}')
print('✅ Test data loaded')

Loading: C:\IIoT-Predictive-Maintenance\processed\FD003_tensors.npz
X_test : (100, 30, 18)
y_test : min=6  max=125  mean=73.8
✅ Test data loaded


In [7]:
# ── FP32 baseline ─────────────────────────────────────────────────────────────
print('Running FP32 baseline ...')
fp32_size  = get_size_mb(fp32_model)
fp32_time  = benchmark_ms(fp32_model, X_test)
fp32_preds = predict(fp32_model, X_test) * max_rul
fp32_rmse  = np.sqrt(mean_squared_error(y_test, fp32_preds))
fp32_mae   = mean_absolute_error(y_test, fp32_preds)
fp32_nasa  = nasa_score(y_test, fp32_preds)
print(f'  Size: {fp32_size:.3f} MB  |  Time: {fp32_time:.1f} ms  |  RMSE: {fp32_rmse:.4f}  |  NASA: {fp32_nasa:.1f}')

Running FP32 baseline ...
  Size: 0.433 MB  |  Time: 596.9 ms  |  RMSE: 14.2275  |  NASA: 381.7


In [8]:
# ── Apply Dynamic Quantization ────────────────────────────────────────────────
# Targets: nn.GRU + nn.LSTM + nn.Linear  →  weights INT8
print('Applying dynamic quantization (GRU + LSTM + Linear → INT8) ...')

int8_model = torch.quantization.quantize_dynamic(
    copy.deepcopy(fp32_model).cpu().eval(),
    qconfig_spec={nn.GRU, nn.LSTM, nn.Linear},
    dtype=torch.qint8
)

int8_size  = get_size_mb(int8_model)
int8_time  = benchmark_ms(int8_model, X_test)
int8_preds = predict(int8_model, X_test) * max_rul
int8_rmse  = np.sqrt(mean_squared_error(y_test, int8_preds))
int8_mae   = mean_absolute_error(y_test, int8_preds)
int8_nasa  = nasa_score(y_test, int8_preds)
print(f'  Size: {int8_size:.3f} MB  |  Time: {int8_time:.1f} ms  |  RMSE: {int8_rmse:.4f}  |  NASA: {int8_nasa:.1f}')

Applying dynamic quantization (GRU + LSTM + Linear → INT8) ...
  Size: 0.123 MB  |  Time: 119.2 ms  |  RMSE: 14.3528  |  NASA: 401.6


In [9]:
# ── Results table ─────────────────────────────────────────────────────────────
size_reduction = (1 - int8_size / fp32_size) * 100
speedup        = fp32_time / int8_time
rmse_delta     = (int8_rmse - fp32_rmse) / fp32_rmse * 100
mae_delta      = (int8_mae  - fp32_mae)  / fp32_mae  * 100
nasa_delta     = (int8_nasa - fp32_nasa) / abs(fp32_nasa) * 100

print(f'\n{"="*60}')
print(f'  GRU-LSTM Quantization Report  |  {DATASET}')
print(f'{"─"*60}')
print(f'  {"Metric":<24} {"FP32":>10} {"INT8":>10} {"Δ":>10}')
print(f'{"─"*60}')
print(f'  {"Model size (MB)":<24} {fp32_size:>10.3f} {int8_size:>10.3f} {-size_reduction:>+9.1f}%')
print(f'  {"Inference (ms)":<24} {fp32_time:>10.1f} {int8_time:>10.1f} {speedup:>9.2f}x')
print(f'  {"RMSE (cycles)":<24} {fp32_rmse:>10.4f} {int8_rmse:>10.4f} {rmse_delta:>+9.2f}%')
print(f'  {"MAE  (cycles)":<24} {fp32_mae:>10.4f} {int8_mae:>10.4f} {mae_delta:>+9.2f}%')
print(f'  {"NASA Score":<24} {fp32_nasa:>10.1f} {int8_nasa:>10.1f} {nasa_delta:>+9.2f}%')
print(f'{"─"*60}')

THRESHOLD = 5.0
if rmse_delta <= THRESHOLD:
    print(f'\n  ✅  PTQ SUCCESSFUL — RMSE degraded only {rmse_delta:+.2f}%')
    save_path = f'{DATA_DIR}\\gru_lstm_int8_{DATASET}.pt'
    torch.save(int8_model, save_path)   # save full object, not state_dict
    print(f'  INT8 model saved → {save_path}')
    print(f'  Reload: model = torch.load(save_path); preds = model(X) * {max_rul}')
else:
    print(f'\n  ⚠️   PTQ FAILED — RMSE degraded {rmse_delta:+.2f}% (limit: {THRESHOLD}%)')
    print('  → Upgrade to Quantization-Aware Training (QAT)')

print(f'{"="*60}')


  GRU-LSTM Quantization Report  |  FD003
────────────────────────────────────────────────────────────
  Metric                         FP32       INT8          Δ
────────────────────────────────────────────────────────────
  Model size (MB)               0.433      0.123     -71.7%
  Inference (ms)                596.9      119.2      5.01x
  RMSE (cycles)               14.2275    14.3528     +0.88%
  MAE  (cycles)               10.4709    10.5803     +1.05%
  NASA Score                    381.7      401.6     +5.22%
────────────────────────────────────────────────────────────

  ✅  PTQ SUCCESSFUL — RMSE degraded only +0.88%
  INT8 model saved → C:\IIoT-Predictive-Maintenance\processed\gru_lstm_int8_FD003.pt
  Reload: model = torch.load(save_path); preds = model(X) * 125.0
